#  Нейронные сети: Автокодировщик, Генеративно-Состязательные сети

## Библиотеки

In [111]:
from copy import deepcopy

import matplotlib.pyplot as plt
from matplotlib.image import imread
from mpl_toolkits import mplot3d
from matplotlib import gridspec
from PIL import Image
import io
import os
from urllib.request import urlopen
from skimage.segmentation import mark_boundaries

from tqdm.notebook import tqdm
import numpy as np
import requests
from scipy.stats import norm
import torch

from sklearn.metrics import classification_report
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms


In [112]:
import warnings
import math
warnings.filterwarnings("ignore")

In [113]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

# Tensorboard

In [114]:
%load_ext tensorboard
%tensorboard --logdir ./

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


## Код для обучени модели

In [115]:
def train_epoch(train_generator, model, loss_function, optimizer, callback = None):
    epoch_loss = 0
    total = 0
    for it, (batch_of_x, batch_of_y) in enumerate(train_generator):
        batch_loss = train_on_batch(model, batch_of_x, batch_of_y, optimizer, loss_function)
        
        if callback is not None:
            with torch.no_grad():
                callback(model, batch_loss)
            
        epoch_loss += batch_loss*len(batch_of_x)
        total += len(batch_of_x)
    
    return epoch_loss/total
        

In [116]:
def trainer(count_of_epoch, 
            batch_size, 
            dataset,
            model, 
            loss_function,
            optimizer,
            lr = 0.001,
            callback = None):

    optima = optimizer(model.parameters(), lr=lr)
    
    iterations = tqdm(range(count_of_epoch), desc='epoch')
    iterations.set_postfix({'train epoch loss': np.nan})
    for it in iterations:
        batch_generator = tqdm(
            torch.utils.data.DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True), 
            leave=False, total=len(dataset)//batch_size+(len(dataset)%batch_size> 0))
        
        epoch_loss = train_epoch(train_generator=batch_generator, 
                    model=model, 
                    loss_function=loss_function, 
                    optimizer=optima, 
                    callback=callback)
        
        iterations.set_postfix({'train epoch loss': epoch_loss})

## Автокодировщик

In [117]:
digit_size = (28, 28)

### Код для обучения модели автокодировщика

In [118]:
def train_on_batch(model, x_batch, y_batch, optimizer, loss_function):
    model.train()
    optimizer.zero_grad()
    
    output = model(x_batch.to(model.device))
    
    loss = loss_function(output, x_batch.to(model.device))
    loss.backward()

    optimizer.step()
    return loss.cpu().item()

In [119]:
class callback():
    def __init__(self, writer, dataset, loss_function, delimeter = 100, batch_size=64):
        self.step = 0
        self.writer = writer
        self.delimeter = delimeter
        self.loss_function = loss_function
        self.batch_size = batch_size

        self.dataset = dataset

    def forward(self, model, loss):
        self.step += 1
        self.writer.add_scalar('LOSS/train', loss, self.step)
        
        if self.step % self.delimeter == 0:
            
            batch_generator = torch.utils.data.DataLoader(dataset = self.dataset, 
                                                          batch_size=self.batch_size)
            
            pred = []
            real = []
            test_loss = 0
            model.eval()
            for it, (x_batch, _) in enumerate(batch_generator):
                x_batch = x_batch.to(model.device)

                output = model(x_batch)

                test_loss += self.loss_function(output, x_batch).cpu().item()*len(x_batch)

                pred.extend(torch.argmax(output, dim=-1).cpu().numpy().tolist())
            
            test_loss /= len(self.dataset)
            
            self.writer.add_scalar('LOSS/test', test_loss, self.step)
          
    def __call__(self, model, loss):
        return self.forward(model, loss)

### Вариационный автокодировщик

#### Модель

In [120]:
class VAE(torch.nn.Module):
    @property
    def device(self):
        return next(self.parameters()).device

    def __init__(self, latent_dim, input_dim, hidden_dim=200):
        """
        Standard VAE with Gaussian observation model.
        Args:
            latent_dim: int - dimension of latent space z.
            input_dim: int - dimension of input space x.
            hidden_dim: int - number of hidden units in encoder/decoder.
        """
        super(VAE, self).__init__()
        self.latent_dim = latent_dim
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        # Encoder
        self.proposal_z = torch.nn.Sequential(
            torch.nn.Linear(self.input_dim, hidden_dim),
            torch.nn.LeakyReLU(),
        )
        self.proposal_mu = torch.nn.Linear(hidden_dim, self.latent_dim)
        self.proposal_sigma = torch.nn.Linear(hidden_dim, self.latent_dim)

        # Decoder
        self.generative_network = torch.nn.Sequential(
            torch.nn.Linear(self.latent_dim, hidden_dim),
            torch.nn.LeakyReLU()
        )
        self.decoder_mu = torch.nn.Linear(hidden_dim, input_dim)
        self.decoder_logvar = torch.nn.Linear(hidden_dim, input_dim)

    def q_z(self, x):
        """
        Returns parameters of q(z|x) = N(mu, sigma^2 I).
        Args:
            x: Tensor of shape (batch_size, input_dim)
        Returns:
            mu: Tensor shape (batch_size, latent_dim)
            sigma: Tensor shape (batch_size, latent_dim) (positive)
        """
        x = x.to(self.device)
        h = self.proposal_z(x)
        mu = self.proposal_mu(h)
        sigma = torch.nn.Softplus()(self.proposal_sigma(h))
        return mu, sigma

    def p_z(self, num_samples):
        """
        Returns parameters of prior distribution p(z) = N(0, I).
        Args:
            num_samples: int
        Returns:
            mu: Tensor shape (num_samples, latent_dim) (all zeros)
            sigma: Tensor shape (num_samples, latent_dim) (all ones)
        """
        mu = torch.zeros([num_samples, self.latent_dim], device=self.device)
        sigma = torch.ones([num_samples, self.latent_dim], device=self.device)
        return mu, sigma

    def sample_z(self, distr, num_samples=1):
        """
        Reparameterization trick: sample z ~ N(mu, sigma^2 I).
        Args:
            distr: tuple (mu, sigma) each shape (batch_size, latent_dim)
            num_samples: number of samples per batch element
        Returns:
            Tensor shape (batch_size, num_samples, latent_dim)
        """
        mu, sigma = distr
        mu = mu.to(self.device)
        sigma = sigma.to(self.device)
        batch_size = mu.shape[0]

        bias = mu.view(batch_size, 1, self.latent_dim)
        epsilon = torch.randn([batch_size, num_samples, self.latent_dim],
                              requires_grad=True,
                              device=self.device)
        scale = sigma.view(batch_size, 1, self.latent_dim)
        return bias + epsilon * scale

    def q_x(self, z):
        """
        Decoder: given z, returns parameters of p(x|z) = N(mu_x, diag(sigma_x^2)).
        Args:
            z: Tensor shape (batch_size, num_samples, latent_dim)
        Returns:
            mu_x: Tensor shape (batch_size, num_samples, input_dim)
            logvar_x: Tensor shape (batch_size, num_samples, input_dim)
        """
        h = self.generative_network(z)
        mu = self.decoder_mu(h)
        logvar = self.decoder_logvar(h)
        return mu, logvar

    def loss(self, batch_x, batch_y):
        """
        Negative ELBO for Gaussian output.
        Args:
            batch_x: Tensor shape (batch_size, input_dim)
            batch_y: dummy, not used
        Returns:
            scalar loss (negative ELBO)
        """
        batch_x = batch_x.to(self.device)
        batch_size = batch_x.shape[0]

        propos_distr = self.q_z(batch_x)
        pri_distr = self.p_z(batch_size)

        z = self.sample_z(propos_distr)
        mu_x, logvar_x = self.q_x(z)
        log_lik = self.gaussian_log_likelihood(batch_x, mu_x, logvar_x)
        expectation = torch.mean(self.log_mean_exp(log_lik), dim=0)

        divergence = self.divergence_KL_normal(propos_distr, pri_distr)

        return -1 * torch.mean(expectation - divergence, dim=0)

    def generate_samples(self, num_samples):
        """
        Generate new samples from the learned generative model.
        Args:
            num_samples: int
        Returns:
            Tensor shape (num_samples, input_dim) – sampled x.
        """

        mu_z, sigma_z = self.p_z(num_samples=1)
        z = self.sample_z((mu_z, sigma_z), num_samples=num_samples)
        mu_x, logvar_x = self.q_x(z)
        std_x = torch.exp(0.5 * logvar_x)
        eps = torch.randn_like(mu_x)
        x_sample = mu_x + eps * std_x
        return x_sample.view(num_samples, -1)

    @staticmethod
    def gaussian_log_likelihood(x_true, mu_x, logvar_x):
        """
        Compute log-likelihood of data under a Gaussian distribution with diagonal covariance.
        Args:
            x_true: Tensor shape (batch_size, input_dim)
            mu_x: Tensor shape (batch_size, num_samples, input_dim)
            logvar_x: Tensor shape (batch_size, num_samples, input_dim)
        Returns:
            Tensor shape (batch_size, num_samples) – log-likelihood for each sample.
        """

        x_true = x_true.unsqueeze(1)
        var = torch.exp(logvar_x)
        log_prob = -0.5 * ((x_true - mu_x) ** 2 / var + logvar_x + math.log(2 * math.pi))
        return torch.sum(log_prob, dim=-1)

    @staticmethod
    def log_mean_exp(data):
        """
        Stable log(mean(exp(data))) along the last dimension.
        Args:
            data: Tensor of arbitrary shape, last dimension is the "sample" dimension.
        Returns:
            Tensor of shape (*data.shape[:-1]) – log-mean-exp.
        """
        return torch.logsumexp(data, dim=-1) - torch.log(torch.Tensor([data.shape[-1]]).to(data.device))

    @staticmethod
    def divergence_KL_normal(q_distr, p_distr):
        """
        KL divergence between two diagonal Gaussian distributions.
        q_distr = (mu_q, sigma_q), p_distr = (mu_p, sigma_p).
        Returns: KL(q || p) for each batch element.
        """
        q_mu, q_sigma = q_distr
        p_mu, p_sigma = p_distr
        D_KL = torch.sum((q_sigma / p_sigma) ** 2, dim=1)
        D_KL -= p_mu.shape[1]
        D_KL += 2 * torch.sum(torch.log(p_sigma), dim=1) - 2 * torch.sum(torch.log(q_sigma), dim=1)
        D_KL += torch.sum((p_mu - q_mu) ** 2 / (p_sigma ** 2), dim=1)
        return 0.5 * D_KL

    def forward(self, x):
        """
        Reconstruct input by returning the mean of p(x|z).
        Args:
            x: Tensor shape (batch_size, input_dim)
        Returns:
            Tensor shape (batch_size, input_dim) – reconstructed mean.
        """
        mu_z, sigma_z = self.q_z(x)
        z = self.sample_z((mu_z, sigma_z), num_samples=1)
        mu_x, _ = self.q_x(z)
        return mu_x.view_as(x)

#### Скрипты для обучение VAE

In [121]:
def train_on_batch(model, x_batch, y_batch, optimizer, loss_function):
    model.train()
    optimizer.zero_grad()
    
    loss = model.loss(x_batch.to(model.device), y_batch.to(model.device))
    loss.backward()

    optimizer.step()
    return loss.cpu().item()

#### Данные

In [122]:
def generate_gaussian_clusters(n_clusters=5, n_points_per_cluster=500, dim=2, 
                               means=None, covs=None, random_state=42):
    """
    Генерирует выборку из нескольких гауссовских кластеров.
    
    Параметры:
        n_clusters (int): количество кластеров
        n_points_per_cluster (int): количество точек в каждом кластере
        dim (int): размерность пространства
        means (list of np.ndarray): список средних для каждого кластера (если None, генерируются случайно)
        covs (list of np.ndarray): список ковариационных матриц (если None, генерируются случайные диагональные)
        random_state (int): seed для воспроизводимости
        
    Возвращает:
        X (np.ndarray): массив формы (n_points_total, dim)
        labels (np.ndarray): массив меток кластеров (0..n_clusters-1)
    """
    np.random.seed(random_state)
    
    if means is None:
        # Генерируем случайные средние в диапазоне [-5, 5] для каждого кластера
        means = [np.random.uniform(-5, 5, size=dim) for _ in range(n_clusters)]
    
    if covs is None:
        # Генерируем случайные диагональные ковариационные матрицы
        covs = []
        for _ in range(n_clusters):
            # Диагональные элементы от 0.5 до 2.0
            diag = np.random.uniform(0.5, 2.0, size=dim)
            cov = np.diag(diag)
            covs.append(cov)
    
    X_list = []
    labels_list = []
    for i, (mu, cov) in enumerate(zip(means, covs)):
        # Генерация точек из многомерного нормального распределения
        X_cluster = np.random.multivariate_normal(mu, cov, size=n_points_per_cluster)
        X_list.append(X_cluster)
        labels_list.append(np.full(n_points_per_cluster, i))
    
    X = np.vstack(X_list)
    labels = np.hstack(labels_list)
    
    return X, labels, means, cov

#### Инициализация модели

In [123]:
optimizer = torch.optim.Adam
loss_function = torch.nn.MSELoss()

#### Перебор разного размера латентного пространства

In [124]:
for dim in [32, 128, 256]:
    n_clusters=3
    n_points_per_cluster=10000
    X, labels, means, covs = generate_gaussian_clusters(n_clusters, n_points_per_cluster, dim)
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X, dtype=torch.float32)
    
    X = TensorDataset(X, X)
    for d in [dim // 8, dim // 4, dim // 2]:
        autoencoder = VAE(input_dim=dim, latent_dim=dim, hidden_dim=d)
        autoencoder.to(device)
    
        writer = SummaryWriter(log_dir = f'autoencoder-vae/all/{d}+x_dim{dim}') #/{d}+x_dim{dim}
        call = callback(writer, X, loss_function, delimeter = 100)
    
        trainer(count_of_epoch=10, 
                batch_size=64, 
                dataset=X,
                model=autoencoder, 
                loss_function=None,
                optimizer = optimizer,
                lr = 0.001,
                callback = call)

epoch:   0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

...

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

In [3]:
!kill $(pgrep tensorboard)
!tensorboard --logdir autoencoder-vae/all

kill: использование: kill [-s назв_сигнала | -n номер_сигнала | -назв_сигнала] ид_процесса | назв_задания] ... или kill -l [назв_сигнала]
/home/sasha/Documents/venv/lib/python3.12/site-packages/tensorboard/default.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-03-26 21:57:31.786573: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Serving 

**Вывод** лучше всего себя показала модель 16 - размерность срытого 32 размерность предсказываемого 

In [126]:
class VAE_1(torch.nn.Module):
    @property
    def device(self):
        return next(self.parameters()).device

    def __init__(self, latent_dim, input_dim, hidden_dim=200):
        """
        Standard VAE with Gaussian observation model.
        Args:
            latent_dim: int - dimension of latent space z.
            input_dim: int - dimension of input space x.
            hidden_dim: int - number of hidden units in encoder/decoder.
        """
        super(VAE_1, self).__init__()
        self.latent_dim = latent_dim
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        new_lay_dim = (self.input_dim + self.hidden_dim) // 2
        # Encoder
        self.proposal_z = torch.nn.Sequential(
            torch.nn.Linear(self.input_dim, new_lay_dim),
            torch.nn.LeakyReLU(),
            torch.nn.Linear(new_lay_dim, self.hidden_dim),
            torch.nn.LeakyReLU(),
        )
        self.proposal_mu = torch.nn.Linear(self.hidden_dim, self.latent_dim)
        self.proposal_sigma = torch.nn.Linear(self.hidden_dim, self.latent_dim)

        # Decoder
        self.generative_network = torch.nn.Sequential(
            torch.nn.Linear(self.latent_dim, new_lay_dim),
            torch.nn.LeakyReLU(),
            torch.nn.Linear(new_lay_dim, self.hidden_dim),
            torch.nn.LeakyReLU(),
        )
        self.decoder_mu = torch.nn.Linear(self.hidden_dim, self.input_dim)
        self.decoder_logvar = torch.nn.Linear(self.hidden_dim, self.input_dim)

    def q_z(self, x):
        """
        Returns parameters of q(z|x) = N(mu, sigma^2 I).
        Args:
            x: Tensor of shape (batch_size, input_dim)
        Returns:
            mu: Tensor shape (batch_size, latent_dim)
            sigma: Tensor shape (batch_size, latent_dim) (positive)
        """
        x = x.to(self.device)
        h = self.proposal_z(x)
        mu = self.proposal_mu(h)
        sigma = torch.nn.Softplus()(self.proposal_sigma(h))
        return mu, sigma

    def p_z(self, num_samples):
        """
        Returns parameters of prior distribution p(z) = N(0, I).
        Args:
            num_samples: int
        Returns:
            mu: Tensor shape (num_samples, latent_dim) (all zeros)
            sigma: Tensor shape (num_samples, latent_dim) (all ones)
        """
        mu = torch.zeros([num_samples, self.latent_dim], device=self.device)
        sigma = torch.ones([num_samples, self.latent_dim], device=self.device)
        return mu, sigma

    def sample_z(self, distr, num_samples=1):
        """
        Reparameterization trick: sample z ~ N(mu, sigma^2 I).
        Args:
            distr: tuple (mu, sigma) each shape (batch_size, latent_dim)
            num_samples: number of samples per batch element
        Returns:
            Tensor shape (batch_size, num_samples, latent_dim)
        """
        mu, sigma = distr
        mu = mu.to(self.device)
        sigma = sigma.to(self.device)
        batch_size = mu.shape[0]

        bias = mu.view(batch_size, 1, self.latent_dim)
        epsilon = torch.randn([batch_size, num_samples, self.latent_dim],
                              requires_grad=True,
                              device=self.device)
        scale = sigma.view(batch_size, 1, self.latent_dim)
        return bias + epsilon * scale

    def q_x(self, z):
        """
        Decoder: given z, returns parameters of p(x|z) = N(mu_x, diag(sigma_x^2)).
        Args:
            z: Tensor shape (batch_size, num_samples, latent_dim)
        Returns:
            mu_x: Tensor shape (batch_size, num_samples, input_dim)
            logvar_x: Tensor shape (batch_size, num_samples, input_dim)
        """
        h = self.generative_network(z)
        mu = self.decoder_mu(h)
        logvar = self.decoder_logvar(h)
        return mu, logvar

    def loss(self, batch_x, batch_y):
        """
        Negative ELBO for Gaussian output.
        Args:
            batch_x: Tensor shape (batch_size, input_dim)
            batch_y: dummy, not used
        Returns:
            scalar loss (negative ELBO)
        """
        batch_x = batch_x.to(self.device)
        batch_size = batch_x.shape[0]

        propos_distr = self.q_z(batch_x)
        pri_distr = self.p_z(batch_size)

        z = self.sample_z(propos_distr)
        mu_x, logvar_x = self.q_x(z)
        log_lik = self.gaussian_log_likelihood(batch_x, mu_x, logvar_x)
        expectation = torch.mean(self.log_mean_exp(log_lik), dim=0)

        divergence = self.divergence_KL_normal(propos_distr, pri_distr)

        return -1 * torch.mean(expectation - divergence, dim=0)

    def generate_samples(self, num_samples):
        """
        Generate new samples from the learned generative model.
        Args:
            num_samples: int
        Returns:
            Tensor shape (num_samples, input_dim) – sampled x.
        """

        mu_z, sigma_z = self.p_z(num_samples=1)
        z = self.sample_z((mu_z, sigma_z), num_samples=num_samples)
        mu_x, logvar_x = self.q_x(z)
        std_x = torch.exp(0.5 * logvar_x)
        eps = torch.randn_like(mu_x)
        x_sample = mu_x + eps * std_x
        return x_sample.view(num_samples, -1)

    @staticmethod
    def gaussian_log_likelihood(x_true, mu_x, logvar_x):
        """
        Compute log-likelihood of data under a Gaussian distribution with diagonal covariance.
        Args:
            x_true: Tensor shape (batch_size, input_dim)
            mu_x: Tensor shape (batch_size, num_samples, input_dim)
            logvar_x: Tensor shape (batch_size, num_samples, input_dim)
        Returns:
            Tensor shape (batch_size, num_samples) – log-likelihood for each sample.
        """

        x_true = x_true.unsqueeze(1)
        var = torch.exp(logvar_x)
        log_prob = -0.5 * ((x_true - mu_x) ** 2 / var + logvar_x + math.log(2 * math.pi))
        return torch.sum(log_prob, dim=-1)

    @staticmethod
    def log_mean_exp(data):
        """
        Stable log(mean(exp(data))) along the last dimension.
        Args:
            data: Tensor of arbitrary shape, last dimension is the "sample" dimension.
        Returns:
            Tensor of shape (*data.shape[:-1]) – log-mean-exp.
        """
        return torch.logsumexp(data, dim=-1) - torch.log(torch.Tensor([data.shape[-1]]).to(data.device))

    @staticmethod
    def divergence_KL_normal(q_distr, p_distr):
        """
        KL divergence between two diagonal Gaussian distributions.
        q_distr = (mu_q, sigma_q), p_distr = (mu_p, sigma_p).
        Returns: KL(q || p) for each batch element.
        """
        q_mu, q_sigma = q_distr
        p_mu, p_sigma = p_distr
        D_KL = torch.sum((q_sigma / p_sigma) ** 2, dim=1)
        D_KL -= p_mu.shape[1]
        D_KL += 2 * torch.sum(torch.log(p_sigma), dim=1) - 2 * torch.sum(torch.log(q_sigma), dim=1)
        D_KL += torch.sum((p_mu - q_mu) ** 2 / (p_sigma ** 2), dim=1)
        return 0.5 * D_KL

    def forward(self, x):
        """
        Reconstruct input by returning the mean of p(x|z).
        Args:
            x: Tensor shape (batch_size, input_dim)
        Returns:
            Tensor shape (batch_size, input_dim) – reconstructed mean.
        """
        mu_z, sigma_z = self.q_z(x)
        z = self.sample_z((mu_z, sigma_z), num_samples=1)
        mu_x, _ = self.q_x(z)
        return mu_x.view_as(x)

In [127]:
optimizer = torch.optim.Adam
loss_function = torch.nn.MSELoss()

In [128]:
for dim in [32, 128, 256]:
    n_clusters=3
    n_points_per_cluster=10000
    X, labels, means, covs = generate_gaussian_clusters(n_clusters, n_points_per_cluster, dim)
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X, dtype=torch.float32)
    
    X = TensorDataset(X, X)
    for d in [dim // 8, dim // 4, dim // 2]:
        autoencoder = VAE_1(input_dim=dim, latent_dim=dim, hidden_dim=d)
        autoencoder.to(device)
    
        writer = SummaryWriter(log_dir = f'autoencoder-vae/all_1/{d}+x_dim{dim}') #/{d}+x_dim{dim}
        call = callback(writer, X, loss_function, delimeter = 100)
    
        trainer(count_of_epoch=10, 
                batch_size=64, 
                dataset=X,
                model=autoencoder, 
                loss_function=None,
                optimizer = optimizer,
                lr = 0.001,
                callback = call)

epoch:   0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

...

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

  0%|          | 0/469 [00:00<?, ?it/s]

In [2]:
!kill $(pgrep tensorboard)
!tensorboard --logdir autoencoder-vae/all_1

kill: использование: kill [-s назв_сигнала | -n номер_сигнала | -назв_сигнала] ид_процесса | назв_задания] ... или kill -l [назв_сигнала]
/home/sasha/Documents/venv/lib/python3.12/site-packages/tensorboard/default.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-03-26 21:54:23.281446: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Serving 

**Вывод** лучше всего себя показала модель 16 - размерность срытого 32 размерность предсказываемого 